In [1]:
!pip -q install "transformers>=4.44.2" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.1" datasets pillow pandas torch torchvision --upgrade

!pip install git+https://github.com/huggingface/transformers

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-4nix7ldo
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-4nix7ldo
  Resolved https://github.com/huggingface/transformers to commit e2e8dbed13c6a8455fd85c15c9fa91c99d609010
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# 구글드라이브 마운트
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 압축 해제
# !unzip "/content/drive/My Drive/250918/data.zip" -d "/content/"

# 라이브러리, 데이터, 설정

In [2]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# from transformers import (
#     AutoModelForVision2Seq,
#     AutoProcessor,
#     BitsAndBytesConfig,
#     get_linear_schedule_with_warmup
# )

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup


# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/home/team036/aichallenge/train.csv")
test_df  = pd.read_csv("/home/team036/aichallenge/test.csv")

# Qwen2.5-VL-3B와 동일한 데이터 크기로 속도 최적화
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
# 양자화 설정 (A100 40GB에 맞게 최적화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # bfloat16으로 변경하여 성능 향상
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델 (최적화된 설정)
# Flash Attention 2 대신 eager attention 사용 (호환성 문제 해결)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    dtype=torch.bfloat16,  # bfloat16 사용
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"  # Flash Attention 대신 eager 사용
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅 (더 효율적인 설정)
lora_config = LoraConfig(
    r=16,  # r을 8에서 16으로 증가하여 성능 향상
    lora_alpha=32,  # alpha도 비례하여 증가
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 모델 컴파일 비활성화 (Qwen3-VL 호환성 문제로 인해)
# if hasattr(torch, 'compile'):
#     print("모델 컴파일 중...")
#     model = torch.compile(model, mode="reduce-overhead")
#     print("모델 컴파일 완료!")
print("모델 컴파일 비활성화 (호환성 문제 해결)")

# 메모리 최적화
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]


trainable params: 33,030,144 || all params: 4,470,845,952 || trainable%: 0.7388
모델 컴파일 비활성화 (호환성 문제 해결)


/home/team036/.local/lib/python3.10/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [4]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
    "Do NOT provide any explanation or extra text. "
    "Always choose the correct answer based on the image and question."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "You must output the correct answer from a, b, c, or d, as a single lowercase letter without any explanation."
    )

In [5]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


In [6]:
# 검증용 데이터 분리
split = int(len(train_df)*0.85)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# A100 40GB에 최적화된 설정 (배치 크기는 크게, 데이터는 적게)
BATCH_SIZE = 8  # A100에서 효율적인 크기
NUM_WORKERS = 4  # 적당한 워커 수
PIN_MEMORY = True  # GPU 전송 최적화
PERSISTENT_WORKERS = True  # 워커 재시작 오버헤드 제거

train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=DataCollator(processor, True), 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=2
)

valid_loader = DataLoader(
    valid_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=DataCollator(processor, True), 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=2
)

print(f"배치 크기: {BATCH_SIZE}")
print(f"훈련 배치 수: {len(train_loader)}")
print(f"검증 배치 수: {len(valid_loader)}")

배치 크기: 8
훈련 배치 수: 22
검증 배치 수: 4


In [8]:
from tqdm.auto import tqdm
import time
import psutil

# 성능 모니터링 함수
def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU 메모리 - 할당: {allocated:.2f}GB, 예약: {reserved:.2f}GB")

# 모델을 device로 이동 (한 번만)
model = model.to(device)
print_gpu_memory()

# A100 40GB에 최적화된 훈련 설정
GRAD_ACCUM = 2  # 적당한 그래디언트 누적
LEARNING_RATE = 2e-4  # 큰 배치에 맞는 학습률
WARMUP_RATIO = 0.05  # 적당한 워밍업

# 옵티마이저 (최적화된 설정)
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE,
    weight_decay=0.01,
    betas=(0.9, 0.95)  # AdamW 최적화
)

# 학습 스케줄러
num_training_steps = 1 * math.ceil(len(train_loader) / GRAD_ACCUM)
warmup_steps = int(num_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=num_training_steps
)

# 스케일러 (bfloat16 최적화)
scaler = torch.amp.GradScaler('cuda', enabled=True)

print(f"훈련 설정:")
print(f"  - 배치 크기: {BATCH_SIZE}")
print(f"  - 그래디언트 누적: {GRAD_ACCUM}")
print(f"  - 유효 배치 크기: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  - 학습률: {LEARNING_RATE}")
print(f"  - 총 훈련 스텝: {num_training_steps}")
print(f"  - 워밍업 스텝: {warmup_steps}")

# 학습 루프
global_step = 0
best_val_loss = float('inf')
start_time = time.time()

for epoch in range(1):
    model.train()
    running = 0.0
    epoch_start = time.time()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    
    for step, batch in enumerate(progress_bar, start=1):
        # 배치를 미리 device로 이동 (한 번에)
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        
        # Mixed precision 훈련
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({
                "loss": f"{avg_loss:.3f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                "step": f"{global_step}/{num_training_steps}"
            })
            running = 0.0
    
    # 마지막 배치 처리
    if step % GRAD_ACCUM != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1} 훈련 완료 - 소요시간: {epoch_time:.2f}초")

    # Validation (최적화된)
    model.eval()
    val_loss = 0.0
    val_steps = 0
    val_start = time.time()
    
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device, non_blocking=True) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    
    val_time = time.time() - val_start
    avg_val_loss = val_loss / val_steps
    print(f"[Epoch {epoch+1}] valid loss: {avg_val_loss:.4f} (소요시간: {val_time:.2f}초)")
    
    # Best 모델만 저장 (덮어쓰기)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        BEST_DIR = "/home/team036/content/Qwen3-VL-4B-Instruct_best"
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print(f"  ✓ Best model updated! (val_loss: {avg_val_loss:.4f})")
    
    # 각 epoch 모델도 백업
    EPOCH_DIR = f"/home/team036/content/Qwen3-VL-4B-Instruct_epoch_{epoch+1}"
    model.save_pretrained(EPOCH_DIR)
    processor.save_pretrained(EPOCH_DIR)

total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Training Complete!")
print(f"총 소요시간: {total_time:.2f}초 ({total_time/60:.2f}분)")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best model saved at: {BEST_DIR}")
print(f"{'='*50}")

# GPU 메모리 정리
torch.cuda.empty_cache()
print_gpu_memory()

GPU 메모리 - 할당: 16.66GB, 예약: 17.39GB
훈련 설정:
  - 배치 크기: 8
  - 그래디언트 누적: 2
  - 유효 배치 크기: 16
  - 학습률: 0.0002
  - 총 훈련 스텝: 11
  - 워밍업 스텝: 0


Epoch 1 [train]:   0%|          | 0/22 [00:00<?, ?batch/s]/home/team036/.local/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Epoch 1 [train]: 100%|██████████| 22/22 [14:22<00:00, 39.22s/batch, loss=2.423, lr=0.00e+00, step=11/11]



Epoch 1 훈련 완료 - 소요시간: 862.93초


Epoch 1 [valid]: 100%|██████████| 4/4 [02:31<00:00, 37.80s/batch]


[Epoch 1] valid loss: 4.5931 (소요시간: 151.19초)
  ✓ Best model updated! (val_loss: 4.5931)

Training Complete!
총 소요시간: 1016.31초 (16.94분)
Best validation loss: 4.5931
Best model saved at: /home/team036/content/Qwen3-VL-4B-Instruct_best
GPU 메모리 - 할당: 17.11GB, 예약: 17.71GB


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))

  # 실제 모델이 어떻게 답변이 오는지 보면서 파싱할때 어떻게 학습시킨내용대로 나오는지 확인해보거


    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")